# Phase 0: Fetching data from Yahoo Finance

In [4]:
from __future__ import annotations

import argparse
import glob as _glob
import logging
import sys
from datetime import date
from pathlib import Path

# __file__ is not defined in interactive environments (Jupyter notebooks, IPython)
# because those run code in an in-memory session, not from a saved script file
# Add a fallback for notebook use: use current working directory to locate project root
try:
    # Original behavior: works when running as a standalone script (.py file)
    _ROOT = Path(__file__).resolve().parent.parent
except NameError:
    # Fallback for notebooks/interactive use: assumes notebook is in project subdir
    # Adjust the number of .parent calls if your notebook is in a deeper subdirectory
    _ROOT = Path.cwd().parent

print(f"Repo root : {_ROOT}")
sys.path.insert(0, str(_ROOT / "src"))

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
logger = logging.getLogger(__name__)


import os  # noqa: E402

from hifi.data.market import MarketDataFetcher  # noqa: E402, download from Yahoo Finance
from hifi.data.storage import write_ohlcv  # noqa: E402, save to a Parqu

Repo root : /Users/alberto/Documents/projects/HiFi


#### Set list of tickets

In [8]:
_NEW_TICKERS = [
    "MSFT", "NVDA", "GOOGL",          # Technology
#     "BAC", "GS",                       # Finance
#     "CVX",                             # Energy
#     "JNJ", "UNH",                      # Healthcare
#     "AMZN", "WMT",                     # Consumer
#     "CAT",                             # Industrial
#     "NEE",                             # Utilities
]
_DATE_FROM = date(2016, 1, 1)
_DATE_TO = date(2023, 6, 30)

In [ ]:
def _market_exists(ticker: str, data_dir: Path) -> bool:
    pattern = str(data_dir / "market" / f"{ticker}_*.parquet")
    return bool(_glob.glob(pattern))


def acquire_phase10_market(data_dir: Path) -> None:
    fetcher = MarketDataFetcher()
    market_dir = data_dir / "market"
    market_dir.mkdir(parents=True, exist_ok=True)

    print(f"\nPhase 10 Market Data  ({_DATE_FROM} to {_DATE_TO})")
    print(f"  New tickers:  {', '.join(_NEW_TICKERS)}")
    print(f"  Destination: {market_dir}")

    for ticker in _NEW_TICKERS:
        if _market_exists(ticker, data_dir):
            print(f"  {ticker:<6}  SKIP (already present)")
            continue
        print(f"  {ticker:<6}  downloading ... ", end="", flush=True)
        try:
            dataset = fetcher.fetch_ohlcv(ticker, _DATE_FROM, _DATE_TO)
            filename = f"{ticker}_{_DATE_FROM}_{_DATE_TO}.parquet"
            write_ohlcv(dataset, market_dir / filename)
            print(f"OK  ({len(dataset.bars)} bars)")
        except Exception as exc:
            print(f"FAILED: {exc}")
            logger.exception("Failed to acquire %s", ticker)

    print("\nDone. Run 'make bootstrap' to re-seed the 15-ticker performance history.")

In [ ]:
def main() -> None:
    parser = argparse.ArgumentParser(
        description="Acquire Phase 10 market Parquet files for 12 new tickers. Idempotent."
    )
    parser.add_argument(
        "--data-dir",
        default=os.environ.get("HIFI_DATA_DIR", str(_ROOT / "data")),
        help="Destination root directory (default: data/).",
    )
    args = parser.parse_args()
    print("Phase 10 Data Acquisition")
    print("=" * 60)
    print(f"Data dir: {args.data_dir}")
    
    acquire_phase10_market(Path(args.data_dir))


if __name__ == "__main__":
    main()